# LeetCode 667 — Beautiful Arrangement II bằng Python

Link đề: https://leetcode.com/problems/beautiful-arrangement-ii/ · Độ khó: **Medium** · Chủ đề: Array, Math, Construction (dựng đáp án)

Skeleton LeetCode:

```python
class Solution:
    def constructArray(self, n: int, k: int) -> List[int]:
        ...
```

Ghi chú: bài có **nhiều đáp án đúng** ("return any of them"), nên mọi test trong notebook dùng **hàm kiểm tra tính hợp lệ** `is_valid(n, k, ans)` thay vì so sánh cứng với một danh sách. Bài không cần cấu trúc tự dựng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: dựng một **hoán vị** của `1..n` sao cho dãy các **hiệu tuyệt đối giữa hai phần tử liền kề** có **đúng `k` giá trị khác nhau**.
2. Nắm các công cụ Python cần thiết: `abs`, ghép cặp liền kề bằng `zip(a, a[1:])`, `set` và `len(set(...))` để đếm giá trị phân biệt, `itertools.permutations`, hai con trỏ `l, r` và tính chẵn lẻ `i % 2`.
3. Đi từ **vét cạn mọi hoán vị** → **quay lui (backtracking) có cắt nhánh** → **dựng đáp án bằng zigzag trong đoạn `1..k+1` rồi nối tăng dần** → **hai con trỏ `l, r` zigzag `k` phần tử đầu rồi điền đơn điệu** (lời giải nộp) → **đệ quy quy nạp (n, k) → (n-1, k-1)**.
4. Hiểu mấu chốt: dãy **zigzag** `1, n, 2, n-1, ...` tạo ra các hiệu **giảm dần từng đơn vị** (`n-1, n-2, ...`) nên mỗi phần tử zigzag thêm đúng **một** hiệu mới; còn đoạn **đơn điệu** chỉ tạo hiệu `1` (đã có), nên không thêm hiệu mới.
5. Tránh các lỗi kinh điển: so sánh đáp án với danh sách cố định, lệch 1 khi đếm số phần tử zigzag, điền phần còn lại sai chiều theo tính chẵn/lẻ của `k`, bỏ sót phần tử (không còn là hoán vị), và dựng sai vì quên rằng `k < n`.

## 2. Đề bài & ví dụ

Cho hai số nguyên `n` và `k`. Hãy dựng một danh sách `answer` gồm `n` số nguyên dương **khác nhau** trong đoạn `1..n` (tức một hoán vị của `1..n`) thỏa:

- Với `answer = [a1, a2, ..., an]`, danh sách `[|a1 - a2|, |a2 - a3|, ..., |a(n-1) - an|]` có **đúng `k` số nguyên khác nhau**.

Trả về `answer`. Nếu có nhiều đáp án hợp lệ, trả về **bất kỳ** đáp án nào.

**Ví dụ 1**

- Input: `n = 3, k = 1`
- Output: `[1,2,3]`
- Giải thích: các hiệu là `[1, 1]`, chỉ có đúng 1 giá trị khác nhau (`1`).

**Ví dụ 2**

- Input: `n = 3, k = 2`
- Output: `[1,3,2]`
- Giải thích: các hiệu là `[2, 1]`, có đúng 2 giá trị khác nhau.

**Ràng buộc:**

- `1 <= k < n <= 10^4`

Chú ý: vì chỉ có `n - 1` hiệu liền kề nên `k` không thể lớn hơn `n - 1` (đó là lý do đề cho `k < n`). Và vì có nhiều đáp án, cách kiểm tra đúng là **kiểm tra tính chất**, không so sánh từng phần tử với đáp án mẫu.

In [1]:
from typing import List
from itertools import permutations
import random
import sys
import time

examples = [(3, 1), (3, 2)]            # chỉ lưu (n, k); đáp án kiểm tra bằng hàm is_valid, không so cứng

def diffs(a):
    """Danh sách các hiệu tuyệt đối giữa hai phần tử liền kề."""
    return [abs(x - y) for x, y in zip(a, a[1:])]

def is_valid(n, k, ans):
    """Hợp lệ nếu ans là hoán vị của 1..n và các hiệu liền kề có đúng k giá trị khác nhau."""
    return sorted(ans) == list(range(1, n + 1)) and len(set(diffs(ans))) == k

# Hai đáp án mẫu của đề phải hợp lệ
assert is_valid(3, 1, [1, 2, 3])
assert is_valid(3, 2, [1, 3, 2])
print("hiệu của [1,2,3]:", diffs([1, 2, 3]), "| của [1,3,2]:", diffs([1, 3, 2]))
# Các đáp án khác cũng hợp lệ -> cần hàm kiểm tra, không so cứng
print("n=3,k=1 còn có [3,2,1]:", is_valid(3, 1, [3, 2, 1]))
print("n=3,k=2 còn có [2,1,3]? hiệu", diffs([2, 1, 3]), "->", is_valid(3, 2, [2, 1, 3]))

hiệu của [1,2,3]: [1, 1] | của [1,3,2]: [2, 1]
n=3,k=1 còn có [3,2,1]: True
n=3,k=2 còn có [2,1,3]? hiệu [1, 2] -> True


## 3. Kiến thức nền

### 3.1. Hiệu tuyệt đối `abs` và các cặp liền kề với `zip(a, a[1:])`

`abs(x - y)` là khoảng cách giữa hai số trên trục số (luôn không âm). Để duyệt các cặp **liền kề** `(a[0], a[1]), (a[1], a[2]), ...` ta ghép `a` với `a[1:]` (chính nó lệch một ô) bằng `zip`: `zip` dừng khi một trong hai dãy hết, nên ta nhận được đúng `n - 1` cặp.

In [2]:
a = [1, 4, 2, 3]
print("a[1:]       =", a[1:])
print("các cặp     =", list(zip(a, a[1:])))
print("các hiệu    =", [abs(x - y) for x, y in zip(a, a[1:])])
print("abs(2 - 7)  =", abs(2 - 7), "| abs(7 - 2) =", abs(7 - 2))

a[1:]       = [4, 2, 3]
các cặp     = [(1, 4), (4, 2), (2, 3)]
các hiệu    = [3, 2, 1]
abs(2 - 7)  = 5 | abs(7 - 2) = 5


### 3.2. `set` để đếm giá trị **khác nhau**

`set` giữ mỗi giá trị đúng một lần, nên `len(set(diffs))` là **số hiệu phân biệt**. Ví dụ `[3, 2, 1, 1]` có 4 phần tử nhưng chỉ 3 giá trị khác nhau.

In [3]:
d = [3, 2, 1, 1]
print(set(d), "-> số giá trị khác nhau:", len(set(d)))
print(len(set(diffs([1, 4, 2, 3]))), len(set(diffs([1, 2, 3, 4]))))

{1, 2, 3} -> số giá trị khác nhau: 3
3 1


### 3.3. Hoán vị và cách kiểm tra "là hoán vị của `1..n`"

Hoán vị của `1..n` là một danh sách chứa **mỗi số từ 1 đến n đúng một lần**. Cách kiểm tra gọn: `sorted(ans) == list(range(1, n + 1))`. `itertools.permutations(range(1, n + 1))` sinh mọi hoán vị (có `n!` cái), dùng được cho `n` nhỏ.

In [4]:
n = 3
perms = list(permutations(range(1, n + 1)))
print(len(perms), "hoán vị:", perms)
print(sorted([2, 3, 1]) == list(range(1, 4)), sorted([2, 2, 1]) == list(range(1, 4)))

6 hoán vị: [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
True False


### 3.4. Hai con trỏ `l`, `r` và tính chẵn/lẻ `i % 2`

Ta dùng hai biến `l = 1` (số nhỏ nhất còn lại) và `r = n` (số lớn nhất còn lại). Lần lượt lấy `l` rồi `r` rồi `l`...: dùng `i % 2 == 0` để chọn "lượt lấy số nhỏ", ngược lại là "lượt lấy số lớn". Mỗi lần lấy xong thì dịch con trỏ vào trong (`l += 1` hoặc `r -= 1`). Kết quả là dãy zigzag `1, n, 2, n-1, 3, ...`.

In [5]:
n = 6
l, r = 1, n
zig = []
for i in range(n):
    if i % 2 == 0:
        zig.append(l)
        l += 1
    else:
        zig.append(r)
        r -= 1
print("zigzag:", zig)
print("hiệu  :", diffs(zig), "-> các hiệu giảm dần 5,4,3,2,1 và đều khác nhau")

zigzag: [1, 6, 2, 5, 3, 4]
hiệu  : [5, 4, 3, 2, 1] -> các hiệu giảm dần 5,4,3,2,1 và đều khác nhau


### 3.5. Đáp án không duy nhất: vì sao phải kiểm tra bằng tính chất

Có thể hơn một hoán vị thỏa đề. Nếu bạn viết `assert ans == [1, 3, 2]` thì một lời giải **đúng** nhưng trả `[2, 1, 3]` sẽ bị chê oan. Cách làm đúng: viết hàm `is_valid` (đã có ở mục 2) và `assert is_valid(n, k, ans)`. Hãy đếm xem với `n = 4` có bao nhiêu hoán vị thỏa từng `k`.

In [6]:
def count_valid(n, k):
    return sum(1 for p in permutations(range(1, n + 1)) if len(set(diffs(p))) == k)

for k in range(1, 4):
    print(f"n=4, k={k}: có {count_valid(4, k)} hoán vị hợp lệ (trên tổng 24)")

n=4, k=1: có 2 hoán vị hợp lệ (trên tổng 24)
n=4, k=2: có 18 hoán vị hợp lệ (trên tổng 24)
n=4, k=3: có 4 hoán vị hợp lệ (trên tổng 24)


## 4. Ý tưởng nền tảng / trực giác

Thử tay với `n = 5` và vài giá trị `k`:

- `k = 1`: mọi hiệu bằng nhau. `[1,2,3,4,5]` cho các hiệu `1,1,1,1` nên chỉ có 1 giá trị. Đây là cách duy nhất "tự nhiên" (cùng đảo ngược `[5,4,3,2,1]`).
- `k = 4 = n - 1`: cần **mọi** hiệu khác nhau (có đúng 4 hiệu). Thử `[1,5,2,4,3]`: hiệu `4,3,2,1` đều khác nhau.
- `k = 2`: cần hai giá trị. Thử `[1,2,3,5,4]`: hiệu `1,1,2,1` → `{1, 2}`: được.

Rút ra hai nhận xét:

1. **Dãy đơn điệu liên tiếp** (tăng hoặc giảm từng bước 1) chỉ tạo hiệu `1`: không làm tăng số hiệu khác nhau.
2. **Dãy zigzag** `1, n, 2, n-1, ...` cho các hiệu `n-1, n-2, n-3, ...` **toàn khác nhau**: mỗi phần tử thêm vào tạo thêm đúng **một** hiệu mới.

Vậy muốn `k` hiệu khác nhau ta chỉ cần: **một phần zigzag để "mua" đủ hiệu**, rồi phần **đơn điệu** để "lấp đầy" các phần tử còn lại mà không tạo hiệu mới. Hãy xem trực tiếp bằng code các hiệu của một dãy zigzag rồi nối đơn điệu.

In [7]:
def show(n, a):
    d = diffs(a)
    print(f"n={n}: {a}\n      hiệu {d} -> {len(set(d))} giá trị khác nhau: {sorted(set(d), reverse=True)}")

show(5, [1, 5, 2, 4, 3])        # zigzag toàn bộ: 4 hiệu khác nhau
show(5, [1, 2, 3, 4, 5])        # đơn điệu: 1 hiệu
show(5, [1, 5, 2, 3, 4])        # zigzag 3 phần tử rồi tăng: 4,3,1,1 -> 3 giá trị
show(6, [1, 3, 2, 4, 5, 6])     # zigzag nhỏ [1,3,2] rồi tăng: 2,1,2,1,1 -> 2 giá trị

n=5: [1, 5, 2, 4, 3]
      hiệu [4, 3, 2, 1] -> 4 giá trị khác nhau: [4, 3, 2, 1]
n=5: [1, 2, 3, 4, 5]
      hiệu [1, 1, 1, 1] -> 1 giá trị khác nhau: [1]
n=5: [1, 5, 2, 3, 4]
      hiệu [4, 3, 1, 1] -> 3 giá trị khác nhau: [4, 3, 1]
n=6: [1, 3, 2, 4, 5, 6]
      hiệu [2, 1, 2, 1, 1] -> 2 giá trị khác nhau: [2, 1]


Cả hai dạng cho thấy cùng một mẫu: **zigzag trước, đơn điệu sau**. Việc còn lại là chọn **độ dài zigzag** và **hướng điền đơn điệu** sao cho khớp đúng `k`, và chứng minh nó đúng.

## 5. Cách 1 — Brute Force: thử mọi hoán vị

### Ý tưởng

Sinh **mọi** hoán vị của `1..n` bằng `itertools.permutations`, với mỗi hoán vị đếm số hiệu khác nhau; hoán vị đầu tiên có đúng `k` thì trả về. Cách này đúng theo định nghĩa, nên rất hợp làm **oracle** cho `n` nhỏ.

### Code

In [8]:
def construct_brute(n, k):
    for p in permutations(range(1, n + 1)):
        if len(set(diffs(p))) == k:
            return list(p)
    return None            # (không xảy ra khi 1 <= k < n)

for n, k in examples:
    ans = construct_brute(n, k)
    print((n, k), "->", ans)
    assert is_valid(n, k, ans)

(3, 1) -> [1, 2, 3]
(3, 2) -> [1, 3, 2]


### Độ phức tạp và nút thắt

- **Time:** có `n!` hoán vị, mỗi hoán vị tốn `O(n)` để tính hiệu: `O(n! · n)`. Với `n = 10` đã hơn `3.6` triệu hoán vị; với `n = 10^4` là không tưởng.
- **Space:** `O(n)`.

Nút thắt: ta **tìm kiếm mù** trong không gian khổng lồ trong khi đáp án có **cấu trúc rất đều**. Hãy đo thời gian để thấy độ bùng nổ.

In [9]:
def timeit_once(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0

# Chọn k = n - 1 là trường hợp khó nhất (hoán vị đầu tiên thỏa nằm khá xa)
for n in (6, 7, 8, 9):
    res, dt = timeit_once(construct_brute, n, n - 1)
    print(f"n = {n}, k = {n - 1}: {dt * 1000:9.1f} ms -> {res}")

n = 6, k = 5:       0.1 ms -> [1, 6, 2, 5, 3, 4]
n = 7, k = 6:       0.9 ms -> [1, 7, 2, 6, 3, 5, 4]
n = 8, k = 7:       6.4 ms -> [1, 8, 2, 7, 3, 6, 4, 5]
n = 9, k = 8:      67.1 ms -> [1, 9, 2, 8, 3, 7, 4, 6, 5]


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** hiệu của dãy zigzag trong một đoạn **liên tiếp** `[x, x + m - 1]` là `m-1, m-2, ..., 1`: toàn bộ các giá trị từ `1` đến `m - 1`, mỗi giá trị một lần. Nên:

- Zigzag trên `k + 1` số liên tiếp (ví dụ `1..k+1`) tạo ra đúng `k` hiệu `k, k-1, ..., 1`: **đúng `k` giá trị khác nhau**.
- Phần còn lại (`k+2..n`) nối vào bằng chuỗi **tăng dần**: các hiệu mới đều `1` (đã có). Chỉ cần kiểm tra **hiệu ở chỗ nối** (giữa phần tử cuối của zigzag và `k + 2`) cũng không vượt ngoài `1..k`.

Ta thử bằng số. Zigzag trên `1..k+1` rồi nối `k+2..n`:

In [10]:
def zigzag_small_then_inc(n, k):
    zig, l, r = [], 1, k + 1
    for i in range(k + 1):
        if i % 2 == 0:
            zig.append(l); l += 1
        else:
            zig.append(r); r -= 1
    return zig + list(range(k + 2, n + 1))

for n, k in [(5, 2), (6, 3), (8, 4), (9, 8), (7, 1)]:
    a = zigzag_small_then_inc(n, k)
    print(f"n={n}, k={k}: {a} | hiệu {diffs(a)} | giá trị khác nhau = {len(set(diffs(a)))}")
    assert is_valid(n, k, a)

n=5, k=2: [1, 3, 2, 4, 5] | hiệu [2, 1, 2, 1] | giá trị khác nhau = 2
n=6, k=3: [1, 4, 2, 3, 5, 6] | hiệu [3, 2, 1, 2, 1] | giá trị khác nhau = 3
n=8, k=4: [1, 5, 2, 4, 3, 6, 7, 8] | hiệu [4, 3, 2, 1, 3, 1, 1] | giá trị khác nhau = 4
n=9, k=8: [1, 9, 2, 8, 3, 7, 4, 6, 5] | hiệu [8, 7, 6, 5, 4, 3, 2, 1] | giá trị khác nhau = 8
n=7, k=1: [1, 2, 3, 4, 5, 6, 7] | hiệu [1, 1, 1, 1, 1, 1] | giá trị khác nhau = 1


Mọi trường hợp đều đúng. Còn một chi tiết: phần tử cuối của đoạn zigzag có thể đứng ở giữa đoạn (không phải biên), vì vậy hiệu tới `k + 2` phải được kiểm tra là `<= k`. Ở mục 10 ta chứng minh điều đó.

## 7. Các cách tiếp theo

Tổng cộng 5 cách:

1. Vét cạn mọi hoán vị (đã làm).
2. **Quay lui (backtracking)** có cắt nhánh khi số hiệu vượt quá `k`.
3. **Zigzag trong đoạn nhỏ `1..k+1` rồi nối tăng dần** (xây theo mục 6).
4. **Hai con trỏ `l, r`**: zigzag `k` phần tử đầu trên toàn dải `1..n`, rồi điền đơn điệu (lời giải nộp).
5. **Đệ quy quy nạp** `(n, k)` → `(n-1, k-1)` bằng phép đối xứng `y -> n + 1 - y`.

### 7.1. Cách 2 — Quay lui có cắt nhánh

Giải thích từng đoạn:

1. Xây dãy từng phần tử; ở mỗi bước thử các số chưa dùng (`used`).
2. Duy trì tập `seen` các hiệu đã xuất hiện. Nếu `len(seen) > k` thì **cắt nhánh** (không thể quay lại giảm số hiệu).
3. Khi đã đủ `n` phần tử, nếu `len(seen) == k` thì tìm được đáp án.
4. Khi quay lui phải **hoàn tác** việc thêm hiệu mới (ghi nhớ hiệu đó có thật sự mới hay không).

Cách này vẫn là hàm mũ trong trường hợp xấu, nhưng chạy nhanh hơn vét cạn rất nhiều vì cắt được nhánh sớm.

In [11]:
def construct_backtrack(n, k):
    used = [False] * (n + 1)
    path = []
    seen = {}                            # hiệu -> số lần xuất hiện hiện tại (để hoàn tác dễ)

    def dfs():
        if len(path) == n:
            return len(seen) == k
        for x in range(1, n + 1):
            if used[x]:
                continue
            d = abs(x - path[-1]) if path else None
            if d is not None:
                seen[d] = seen.get(d, 0) + 1
                if len(seen) > k:        # cắt nhánh: quá nhiều hiệu khác nhau
                    seen[d] -= 1
                    if seen[d] == 0:
                        del seen[d]
                    continue
            used[x] = True
            path.append(x)
            if dfs():
                return True
            path.pop()
            used[x] = False
            if d is not None:
                seen[d] -= 1
                if seen[d] == 0:
                    del seen[d]
        return False

    return list(path) if dfs() else None

for n, k in examples:
    ans = construct_backtrack(n, k)
    print((n, k), "->", ans)
    assert is_valid(n, k, ans)

(3, 1) -> [1, 2, 3]
(3, 2) -> [1, 3, 2]


**Độ phức tạp Cách 2:** Time hàm mũ trong trường hợp xấu (chỉ dùng được với `n` nhỏ, cỡ vài chục là cùng); Space `O(n)`. Nó chủ yếu mang tính tham khảo: cho thấy "thử và cắt nhánh" không bằng nhìn ra cấu trúc.

Hãy so sánh tốc độ với vét cạn ở `n = 9`, `k = 8`.

In [12]:
for n, k in [(8, 7), (9, 8), (10, 9), (12, 11)]:
    r1, t1 = timeit_once(construct_backtrack, n, k)
    assert is_valid(n, k, r1)
    print(f"backtracking n={n}, k={k}: {t1 * 1000:8.2f} ms")

backtracking n=8, k=7:    10.94 ms


backtracking n=9, k=8:    70.46 ms

backtracking n=10, k=9:   553.89 ms


backtracking n=12, k=11: 59009.54 ms


### 7.2. Cách 3 — Zigzag trong đoạn `1..k+1` rồi nối tăng dần

Giải thích từng đoạn:

1. Tạo zigzag trên `k + 1` số `1..k+1`: lần lượt lấy số nhỏ nhất (`l`), số lớn nhất (`r`), số nhỏ nhất,... `k + 1` lần. Tạo ra `k` hiệu `k, k-1, ..., 1`.
2. Nối thêm `k + 2, k + 3, ..., n` theo thứ tự tăng. Các hiệu mới đều `1`.
3. Hiệu ở chỗ nối nằm trong `1..k` (chứng minh ở mục 10), nên tổng số hiệu khác nhau vẫn là `k`.

In [13]:
def construct_small_zigzag(n, k):
    ans = []
    l, r = 1, k + 1
    for i in range(k + 1):
        if i % 2 == 0:
            ans.append(l)
            l += 1
        else:
            ans.append(r)
            r -= 1
    ans.extend(range(k + 2, n + 1))
    return ans

for n, k in examples:
    ans = construct_small_zigzag(n, k)
    print((n, k), "->", ans)
    assert is_valid(n, k, ans)

(3, 1) -> [1, 2, 3]
(3, 2) -> [1, 3, 2]


In [14]:
class Solution3:
    def constructArray(self, n: int, k: int) -> List[int]:
        ans = []
        l, r = 1, k + 1
        for i in range(k + 1):
            if i % 2 == 0:
                ans.append(l)
                l += 1
            else:
                ans.append(r)
                r -= 1
        ans.extend(range(k + 2, n + 1))
        return ans

for n, k in examples:
    assert is_valid(n, k, Solution3().constructArray(n, k))
print("Solution3 OK")

Solution3 OK


**Độ phức tạp Cách 3:** Time `O(n)`; Space `O(n)` cho danh sách kết quả (không tính, bộ nhớ phụ là `O(1)`).

### 7.3. Cách 4 — Hai con trỏ trên toàn dải `1..n` (lời giải nộp)

Ý tưởng khác một chút: zigzag **trên toàn bộ dải** `1..n` nhưng chỉ lấy **`k` phần tử đầu**, rồi điền `n - k` phần tử còn lại **đơn điệu** từ phía con trỏ vừa dừng.

Giải thích từng đoạn:

1. `l = 1`, `r = n`, `ans = []`.
2. `k` lần đầu: `i` chẵn lấy `l` (rồi `l += 1`), `i` lẻ lấy `r` (rồi `r -= 1`). Hiệu liên tiếp là `n-1, n-2, ..., n-k+1`: `k - 1` giá trị khác nhau.
3. `n - k` lần sau điền đơn điệu: nếu `k` chẵn thì phần tử cuối vừa lấy là `r + 1` (lượt lẻ vừa lấy `r`), nên tiếp tục **giảm** (`ans.append(r); r -= 1`); nếu `k` lẻ thì phần tử cuối vừa lấy là `l - 1`, nên tiếp tục **tăng** (`ans.append(l); l += 1`). Hiệu giữa phần tử cuối của zigzag và phần tử đầu tiên của đoạn điền luôn là `1`, rồi các hiệu tiếp theo cũng `1`.
4. Tổng cộng: `k - 1` hiệu zigzag cộng với hiệu `1` (mới vì `n - k + 1 >= 2`) = `k` hiệu khác nhau.

In [15]:
def construct_two_pointers(n, k):
    l, r = 1, n
    ans = []
    for i in range(k):                  # zigzag k phần tử đầu
        if i % 2 == 0:
            ans.append(l)
            l += 1
        else:
            ans.append(r)
            r -= 1
    for i in range(k, n):               # điền phần còn lại đơn điệu, theo chiều con trỏ vừa dừng
        if k % 2 == 0:
            ans.append(r)
            r -= 1
        else:
            ans.append(l)
            l += 1
    return ans

for n, k in examples:
    ans = construct_two_pointers(n, k)
    print((n, k), "->", ans, "| hiệu", diffs(ans))
    assert is_valid(n, k, ans)

(3, 1) -> [1, 2, 3] | hiệu [1, 1]
(3, 2) -> [1, 3, 2] | hiệu [2, 1]


In [16]:
class Solution4:
    def constructArray(self, n: int, k: int) -> List[int]:
        l, r = 1, n
        ans = []
        for i in range(k):
            if i % 2 == 0:
                ans.append(l)
                l += 1
            else:
                ans.append(r)
                r -= 1
        for i in range(k, n):
            if k % 2 == 0:
                ans.append(r)
                r -= 1
            else:
                ans.append(l)
                l += 1
        return ans

for n, k in examples:
    assert is_valid(n, k, Solution4().constructArray(n, k))
print("Solution4 OK")

Solution4 OK


**Độ phức tạp Cách 4:** Time `O(n)`; Space `O(n)` cho kết quả, `O(1)` bộ nhớ phụ.

### 7.4. Cách 5 — Đệ quy quy nạp `(n, k)` → `(n-1, k-1)`

Ý tưởng "quy bài toán về bài nhỏ hơn":

- **Cơ sở:** `k = 1` thì `[1, 2, ..., n]` (mọi hiệu bằng 1).
- **Bước quy nạp:** với `k > 1`, lấy đáp án `y` của `(n-1, k-1)` (là hoán vị của `1..n-1` bắt đầu bằng `1`), **lật đối xứng** từng phần tử `y -> n + 1 - y` để đổi về các số `2..n`, rồi **đặt `1` ở đầu**.

Hiệu tăng thêm: giữa `1` và phần tử đầu tiên `n + 1 - 1 = n` là `n - 1`, lớn hơn mọi hiệu của phần còn lại (vì phép lật đối xứng giữ nguyên các hiệu, mà đáp án con chỉ dùng số trong `1..n-1` nên hiệu `<= n - 2`). Vậy thêm đúng **một hiệu mới** `n - 1`, số hiệu khác nhau tăng từ `k-1` lên `k`.

Cách này thực chất sinh lại dãy zigzag. Nhược điểm: độ sâu đệ quy `k`, thời gian `O(n · k)` (mỗi tầng tạo danh sách mới). Nó có giá trị **chứng minh bằng quy nạp**, không dùng nộp cho `n` lớn.

In [17]:
def construct_recursive(n, k):
    if k == 1:
        return list(range(1, n + 1))
    smaller = construct_recursive(n - 1, k - 1)       # hoán vị của 1..n-1 với k-1 hiệu
    return [1] + [n + 1 - y for y in smaller]         # lật y -> n+1-y rồi đặt 1 ở đầu

for n, k in examples:
    ans = construct_recursive(n, k)
    print((n, k), "->", ans, "| hiệu", diffs(ans))
    assert is_valid(n, k, ans)
print("n=5,k=3 ->", construct_recursive(5, 3), "| n=6,k=4 ->", construct_recursive(6, 4))

(3, 1) -> [1, 2, 3] | hiệu [1, 1]
(3, 2) -> [1, 3, 2] | hiệu [2, 1]
n=5,k=3 -> [1, 5, 2, 3, 4] | n=6,k=4 -> [1, 6, 2, 5, 4, 3]


In [18]:
class Solution5:
    def constructArray(self, n: int, k: int) -> List[int]:
        if k == 1:
            return list(range(1, n + 1))
        smaller = self.constructArray(n - 1, k - 1)
        return [1] + [n + 1 - y for y in smaller]

for n, k in examples:
    assert is_valid(n, k, Solution5().constructArray(n, k))
print("Solution5 OK")

Solution5 OK


**Độ phức tạp Cách 5:** Time `O(n · k)`; Space `O(n)` cho danh sách mỗi tầng và `O(k)` ngăn xếp đệ quy. Với `k` cỡ `10^4` sẽ gặp `RecursionError` ở cấu hình mặc định; chỉ dùng cho `n, k` nhỏ.

Bảng so sánh cả 5 cách đặt ở mục 16.

## 8. TRACE từng bước

Trace Cách 4 trên `n = 6`, `k = 3`:

| bước `i` | giai đoạn | điều kiện | thêm vào | `l` sau | `r` sau | `ans` sau |
|---|---|---|---|---|---|---|
| 0 | zigzag | `i` chẵn | `l = 1` | 2 | 6 | `[1]` |
| 1 | zigzag | `i` lẻ | `r = 6` | 2 | 5 | `[1,6]` |
| 2 | zigzag | `i` chẵn | `l = 2` | 3 | 5 | `[1,6,2]` |
| 3 | điền | `k = 3` lẻ → tăng | `l = 3` | 4 | 5 | `[1,6,2,3]` |
| 4 | điền | tăng | `l = 4` | 5 | 5 | `[1,6,2,3,4]` |
| 5 | điền | tăng | `l = 5` | 6 | 5 | `[1,6,2,3,4,5]` |

Các hiệu: `5, 4, 1, 1, 1` → các giá trị khác nhau `{5, 4, 1}` → `3 = k`. Hàm sau in đúng từng bước, kèm tập hiệu hiện có.

In [19]:
def trace_two_pointers(n, k):
    l, r = 1, n
    ans = []
    print(f"n={n}, k={k}")
    for i in range(n):
        if i < k:
            phase = "zigzag"
            if i % 2 == 0:
                x, l = l, l + 1
            else:
                x, r = r, r - 1
        else:
            phase = "điền  "
            if k % 2 == 0:
                x, r = r, r - 1
            else:
                x, l = l, l + 1
        ans.append(x)
        d = diffs(ans)
        print(f"i={i} {phase} thêm {x} | l={l} r={r} | ans={ans} | hiệu={d} | khác nhau={sorted(set(d), reverse=True)}")
    assert is_valid(n, k, ans)
    print("=> hợp lệ,", len(set(diffs(ans))), "giá trị khác nhau")
    return ans

trace_two_pointers(6, 3)

n=6, k=3
i=0 zigzag thêm 1 | l=2 r=6 | ans=[1] | hiệu=[] | khác nhau=[]
i=1 zigzag thêm 6 | l=2 r=5 | ans=[1, 6] | hiệu=[5] | khác nhau=[5]
i=2 zigzag thêm 2 | l=3 r=5 | ans=[1, 6, 2] | hiệu=[5, 4] | khác nhau=[5, 4]
i=3 điền   thêm 3 | l=4 r=5 | ans=[1, 6, 2, 3] | hiệu=[5, 4, 1] | khác nhau=[5, 4, 1]
i=4 điền   thêm 4 | l=5 r=5 | ans=[1, 6, 2, 3, 4] | hiệu=[5, 4, 1, 1] | khác nhau=[5, 4, 1]
i=5 điền   thêm 5 | l=6 r=5 | ans=[1, 6, 2, 3, 4, 5] | hiệu=[5, 4, 1, 1, 1] | khác nhau=[5, 4, 1]
=> hợp lệ, 3 giá trị khác nhau


[1, 6, 2, 3, 4, 5]

Trace trường hợp `k` chẵn để thấy hướng điền **đổi chiều**: `n = 7`, `k = 4`. Sau 4 phần tử zigzag `[1,7,2,6]` (phần tử cuối vừa lấy từ `r`), ta điền **giảm** từ `r = 5`: `5, 4, 3`.

In [20]:
trace_two_pointers(7, 4)

n=7, k=4
i=0 zigzag thêm 1 | l=2 r=7 | ans=[1] | hiệu=[] | khác nhau=[]
i=1 zigzag thêm 7 | l=2 r=6 | ans=[1, 7] | hiệu=[6] | khác nhau=[6]
i=2 zigzag thêm 2 | l=3 r=6 | ans=[1, 7, 2] | hiệu=[6, 5] | khác nhau=[6, 5]
i=3 zigzag thêm 6 | l=3 r=5 | ans=[1, 7, 2, 6] | hiệu=[6, 5, 4] | khác nhau=[6, 5, 4]
i=4 điền   thêm 5 | l=3 r=4 | ans=[1, 7, 2, 6, 5] | hiệu=[6, 5, 4, 1] | khác nhau=[6, 5, 4, 1]
i=5 điền   thêm 4 | l=3 r=3 | ans=[1, 7, 2, 6, 5, 4] | hiệu=[6, 5, 4, 1, 1] | khác nhau=[6, 5, 4, 1]
i=6 điền   thêm 3 | l=3 r=2 | ans=[1, 7, 2, 6, 5, 4, 3] | hiệu=[6, 5, 4, 1, 1, 1] | khác nhau=[6, 5, 4, 1]
=> hợp lệ, 4 giá trị khác nhau


[1, 7, 2, 6, 5, 4, 3]

Trace Cách 3 (zigzag nhỏ rồi nối tăng) trên `n = 7`, `k = 3`: zigzag trên `1..4` cho `[1,4,2,3]` với hiệu `3, 2, 1`, rồi nối `5, 6, 7`. Hiệu ở chỗ nối là `|3 - 5| = 2` (đã có), sau đó `1, 1`.

In [21]:
def trace_small_zigzag(n, k):
    ans, l, r = [], 1, k + 1
    print(f"n={n}, k={k}: zigzag trên 1..{k + 1}")
    for i in range(k + 1):
        if i % 2 == 0:
            ans.append(l); l += 1
        else:
            ans.append(r); r -= 1
        print(f"  i={i}: ans={ans} | hiệu={diffs(ans)}")
    for x in range(k + 2, n + 1):
        ans.append(x)
        print(f"  nối {x}: hiệu mới = {abs(ans[-1] - ans[-2])} | tập hiệu = {sorted(set(diffs(ans)), reverse=True)}")
    assert is_valid(n, k, ans)
    return ans

trace_small_zigzag(7, 3)

n=7, k=3: zigzag trên 1..4
  i=0: ans=[1] | hiệu=[]
  i=1: ans=[1, 4] | hiệu=[3]
  i=2: ans=[1, 4, 2] | hiệu=[3, 2]
  i=3: ans=[1, 4, 2, 3] | hiệu=[3, 2, 1]
  nối 5: hiệu mới = 2 | tập hiệu = [3, 2, 1]
  nối 6: hiệu mới = 1 | tập hiệu = [3, 2, 1]
  nối 7: hiệu mới = 1 | tập hiệu = [3, 2, 1]


[1, 4, 2, 3, 5, 6, 7]

## 9. Code nộp LeetCode

Ta chọn **Cách 4 (hai con trỏ `l, r` trên toàn dải)**: ngắn, `O(n)`, không cần chứng minh chỗ nối riêng.

In [22]:
class Solution:
    def constructArray(self, n: int, k: int) -> List[int]:
        l, r = 1, n
        ans = []
        for i in range(k):
            if i % 2 == 0:
                ans.append(l)
                l += 1
            else:
                ans.append(r)
                r -= 1
        for i in range(k, n):
            if k % 2 == 0:
                ans.append(r)
                r -= 1
            else:
                ans.append(l)
                l += 1
        return ans


sol = Solution()
for n, k in examples:
    got = sol.constructArray(n, k)
    print((n, k), "->", got)
    assert is_valid(n, k, got)

(3, 1) -> [1, 2, 3]
(3, 2) -> [1, 3, 2]


## 10. Vì sao đúng

**Phần A — Lời giải nộp (Cách 4).** Đặt `m = n - k` là số phần tử còn lại sau `k` lượt zigzag.

1. **Là hoán vị của `1..n`.** Hai con trỏ `l` (tăng) và `r` (giảm) bắt đầu ở hai đầu; mỗi lần ta lấy đúng một số tại một con trỏ rồi dịch vào trong, nên mỗi số được lấy đúng một lần cho tới khi `l > r`. Tổng số lần lấy là `n` nên lấy hết `1..n`, không trùng, không sót.
2. **Hiệu trong giai đoạn zigzag.** Các phần tử là `1, n, 2, n-1, 3, ...`. Hai phần tử liền kề thứ `i` và `i+1` cách nhau `n - 1 - i` (kiểm tra: `|1 - n| = n-1`, `|n - 2| = n-2`, `|2 - (n-1)| = n-3`, ...). Với `k` phần tử có `k - 1` hiệu: `n-1, n-2, ..., n-k+1`, **đôi một khác nhau**, và đều `>= n - k + 1 >= 2` (vì `k < n`).
3. **Chỗ nối và giai đoạn điền.** Sau `k` lượt, các số chưa dùng là đoạn liên tiếp `[l, r]` với `r - l + 1 = n - k`. Nếu `k` lẻ, phần tử cuối vừa lấy là `l - 1` (lượt chẵn cuối cùng lấy `l`); ta điền `l, l+1, ...` nên hiệu chỗ nối là `1` rồi toàn `1`. Nếu `k` chẵn, phần tử cuối vừa lấy là `r + 1`; ta điền `r, r-1, ...` nên hiệu chỗ nối là `1` rồi toàn `1`. Với `k = 1`: một phần tử `1` rồi `l = 2, 3, ...` điền tăng (vì `k` lẻ) nên toàn hiệu `1`.
4. **Đếm.** Tập hiệu = `{n-1, ..., n-k+1}` (có `k-1` giá trị, đều `>= 2`) hợp với `{1}` (nếu có hiệu nào bằng 1: luôn có vì `n - k >= 1` phần tử điền, tạo ít nhất hiệu chỗ nối). Do `1` không thuộc tập đầu nên tổng đúng `k - 1 + 1 = k` giá trị khác nhau. Khi `k = 1` tập đầu rỗng, tổng `= 1 = k`. ∎

**Phần B — Cách 3.** Zigzag trên `1..k+1` có hiệu `k, k-1, ..., 1`: đúng `k` giá trị. Phần tử cuối của zigzag là phần tử "giữa" của đoạn `1..k+1` nên thuộc `[1, k+1]`; hiệu tới `k + 2` là `k + 2 - e` với `e >= 2` (khi `k >= 2`, vì `e` là trung điểm) nên `<= k`; còn khi `k = 1` dãy là `[1, 2]` rồi `3` với hiệu `1`. Mọi hiệu mới ở đoạn tăng là `1`. Vậy không thêm hiệu ngoài `1..k`, tổng `k`. ∎

**Phần C — Cách 5 (quy nạp).** Giả sử đáp án của `(n-1, k-1)` là hoán vị của `1..n-1` có `k-1` hiệu khác nhau (đều `<= n - 2`). Phép `y -> n + 1 - y` là song ánh từ `1..n-1` lên `2..n` và giữ nguyên mọi hiệu `|y1 - y2|`. Đặt thêm `1` ở đầu ta được hoán vị của `1..n`. Hiệu mới `|1 - (n + 1 - y0)| = n - y0`, trong đó `y0` là phần tử đầu của đáp án con. Với đáp án con luôn bắt đầu bằng `1` (đúng ở cơ sở và được giữ bởi cấu trúc `[1] + ...`), hiệu mới là `n - 1 > n - 2 >=` mọi hiệu cũ, nên thêm đúng một hiệu mới: `k - 1 + 1 = k`. ∎

**Cận của `k`.** Dãy có `n - 1` hiệu nên có tối đa `n - 1` giá trị; `k >= 1` vì `n >= 2` có ít nhất một hiệu. Đề cho `1 <= k < n`, vậy mọi `k` hợp lệ đều dựng được.

## 11. Time / Space Complexity

- **Cách 1 (vét cạn):** Time `O(n! · n)`; Space `O(n)`.
- **Cách 2 (quay lui):** Time hàm mũ trong trường hợp xấu; Space `O(n)`.
- **Cách 3 (zigzag nhỏ + nối tăng):** Time `O(n)`; Space `O(n)` cho kết quả.
- **Cách 4 (hai con trỏ, nộp):** Time `O(n)`; Space `O(n)` cho kết quả, `O(1)` phụ.
- **Cách 5 (đệ quy quy nạp):** Time `O(n · k)`; Space `O(n + k)`.

Ở `n = 10^4` ta đo ba cách tuyến tính (Cách 3, Cách 4) với vài giá trị `k`.

In [23]:
n = 10 ** 4
print(f"n = {n}")
for k in (1, 2, 100, 5000, n - 1):
    out = []
    for name, cls in (("small-zigzag", Solution3), ("two-pointers", Solution4)):
        res, dt = timeit_once(cls().constructArray, n, k)
        assert is_valid(n, k, res)
        out.append(f"{name}: {dt * 1000:6.2f} ms")
    print(f"k = {k:5d} | " + " | ".join(out))

n = 10000
k =     1 | small-zigzag:   0.39 ms | two-pointers:   0.99 ms
k =     2 | small-zigzag:   0.12 ms | two-pointers:   0.87 ms
k =   100 | small-zigzag:   0.14 ms | two-pointers:   1.07 ms
k =  5000 | small-zigzag:   0.45 ms | two-pointers:   0.92 ms
k =  9999 | small-zigzag:   0.94 ms | two-pointers:   0.86 ms


Cả hai chạy trong vài mili-giây ở cỡ tối đa của đề. Vét cạn thì ngay `n = 10` đã vô vọng.

## 12. Bộ test

Bốn lớp test: ví dụ đề, edge case, phản ví dụ cho các cách "gần đúng", và đối chiếu có hệ thống (mọi `(n, k)` nhỏ) cùng ngẫu nhiên có seed. Vì có nhiều đáp án đúng, **mọi kiểm tra dùng `is_valid`**.

In [24]:
all_methods = {
    "small-zigzag": construct_small_zigzag,
    "two-pointers": construct_two_pointers,
    "Sol3": lambda n, k: Solution3().constructArray(n, k),
    "Sol4": lambda n, k: Solution4().constructArray(n, k),
    "submit": lambda n, k: Solution().constructArray(n, k),
}
small_methods = dict(all_methods)
small_methods["brute"] = construct_brute
small_methods["backtrack"] = construct_backtrack
small_methods["recursive"] = construct_recursive

def check_all(n, k, methods):
    for name, fn in methods.items():
        ans = fn(n, k)
        assert ans is not None and is_valid(n, k, ans), (name, n, k, ans)

for n, k in examples:
    check_all(n, k, small_methods)
print("ví dụ đề: OK (mọi cách đều cho đáp án hợp lệ)")

ví dụ đề: OK (mọi cách đều cho đáp án hợp lệ)


In [25]:
# Edge case
edge = [
    (2, 1),            # nhỏ nhất theo ràng buộc: chỉ có một hiệu
    (3, 1), (3, 2),
    (4, 1), (4, 3),    # k = 1 và k = n - 1
    (5, 4),            # zigzag toàn bộ
    (10, 1),           # đơn điệu toàn bộ
    (10, 9),           # k = n - 1
    (10, 2), (10, 8),  # gần hai đầu
]
for n, k in edge:
    check_all(n, k, small_methods if n <= 8 else all_methods)
    print((n, k), "->", Solution().constructArray(n, k))
print("edge case: OK")

(2, 1) -> [1, 2]
(3, 1) -> [1, 2, 3]
(3, 2) -> [1, 3, 2]
(4, 1) -> [1, 2, 3, 4]
(4, 3) -> [1, 4, 2, 3]
(5, 4) -> [1, 5, 2, 4, 3]
(10, 1) -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(10, 9) -> [1, 10, 2, 9, 3, 8, 4, 7, 5, 6]
(10, 2) -> [1, 10, 9, 8, 7, 6, 5, 4, 3, 2]
(10, 8) -> [1, 10, 2, 9, 3, 8, 4, 7, 6, 5]
edge case: OK


Phản ví dụ cho các cách "gần đúng":

- *So sánh cứng với đáp án mẫu*: `[1,3,2]` là đáp án mẫu của `(3, 2)`, nhưng `[2,1,3]` cũng hợp lệ nên `assert ans == [1,3,2]` sẽ sai oan.
- *Dãy tăng dần `1..n`* chỉ đúng với `k = 1`, sai với mọi `k > 1`.
- *Zigzag toàn bộ `1, n, 2, n-1, ...`* chỉ đúng với `k = n - 1`.

In [26]:
print("hợp lệ nhưng khác đáp án mẫu:", is_valid(3, 2, [2, 1, 3]), "| và", is_valid(3, 2, [1, 3, 2]))

def wrong_increasing(n, k):
    return list(range(1, n + 1))

def wrong_full_zigzag(n, k):
    l, r, out = 1, n, []
    for i in range(n):
        if i % 2 == 0:
            out.append(l); l += 1
        else:
            out.append(r); r -= 1
    return out

print("tăng dần với (5, 3):", is_valid(5, 3, wrong_increasing(5, 3)))
print("zigzag toàn bộ với (5, 2):", is_valid(5, 2, wrong_full_zigzag(5, 2)), "| với (5, 4):", is_valid(5, 4, wrong_full_zigzag(5, 4)))
assert not is_valid(5, 3, wrong_increasing(5, 3))
assert not is_valid(5, 2, wrong_full_zigzag(5, 2))
assert is_valid(5, 4, wrong_full_zigzag(5, 4))

hợp lệ nhưng khác đáp án mẫu: True | và True
tăng dần với (5, 3): False
zigzag toàn bộ với (5, 2): False | với (5, 4): True


In [27]:
# Đối chiếu có hệ thống: MỌI (n, k) với 2 <= n <= 8 qua cả 7 cách, và oracle độc lập: tồn tại ít nhất một hoán vị hợp lệ
cnt = 0
for n in range(2, 9):
    for k in range(1, n):
        check_all(n, k, small_methods)
        assert count_valid(n, k) > 0          # oracle: đề luôn có đáp án
        cnt += 1
print("mọi (n, k) với n <= 8:", cnt, "cặp, cả 7 cách đều hợp lệ")

# Mọi (n, k) với n <= 120 qua các cách tuyến tính
cnt = 0
for n in range(2, 121):
    for k in range(1, n):
        check_all(n, k, all_methods)
        cnt += 1
print("mọi (n, k) với n <= 120:", cnt, "cặp, các cách tuyến tính đều hợp lệ")

mọi (n, k) với n <= 8: 28 cặp, cả 7 cách đều hợp lệ


mọi (n, k) với n <= 120: 7140 cặp, các cách tuyến tính đều hợp lệ


In [28]:
# Ngẫu nhiên có seed, đúng ràng buộc 1 <= k < n <= 10^4 (chỉ các cách tuyến tính)
rng = random.Random(2026)
for t in range(300):
    n = rng.randint(2, 10 ** 4)
    k = rng.randint(1, n - 1)
    check_all(n, k, all_methods)
for n in (10 ** 4,):
    for k in (1, 2, 3, n - 2, n - 1):
        check_all(n, k, all_methods)
print("300 test ngẫu nhiên + các biên ở n = 10^4: OK")

# Đệ quy quy nạp: chỉ với n, k nhỏ (độ sâu k)
for n in range(2, 60):
    for k in range(1, n):
        assert is_valid(n, k, construct_recursive(n, k))
print("đệ quy quy nạp (n < 60): OK")

300 test ngẫu nhiên + các biên ở n = 10^4: OK
đệ quy quy nạp (n < 60): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh đáp án với một danh sách cố định

Code sai (test): `assert sol.constructArray(3, 2) == [1, 3, 2]`. Lời giải đúng nhưng ra `[2, 1, 3]` sẽ "fail". Bài có nhiều đáp án nên phải kiểm tra **tính chất** (hoán vị + đúng `k` hiệu).

In [29]:
def another_valid(n, k):
    """Một lời giải hợp lệ nhưng khác đáp án mẫu: lật đối xứng y -> n+1-y của lời giải nộp."""
    return [n + 1 - y for y in Solution().constructArray(n, k)]

ans = another_valid(3, 2)
print("trả về:", ans, "| hợp lệ:", is_valid(3, 2, ans), "| bằng [1,3,2]?", ans == [1, 3, 2])

trả về: [3, 1, 2] | hợp lệ: True | bằng [1,3,2]? False


### Lỗi 2 — Lệch 1 khi đếm số phần tử zigzag

Để có `k` hiệu khác nhau, zigzag cần `k + 1` phần tử (trên đoạn nhỏ `1..k+1`, Cách 3) hoặc `k` phần tử (trên toàn dải, Cách 4 vì còn hiệu `1` từ phần điền). Nếu lẫn hai công thức, kết quả lệch 1 (thừa hoặc thiếu một hiệu).

In [30]:
def buggy_off_by_one(n, k):
    ans, l, r = [], 1, k              # BUG: zigzag trên 1..k (chỉ k số) rồi nối tăng -> thiếu 1 hiệu
    for i in range(k):
        if i % 2 == 0:
            ans.append(l); l += 1
        else:
            ans.append(r); r -= 1
    ans.extend(range(k + 1, n + 1))
    return ans

bad = [(n, k) for n in range(2, 9) for k in range(1, n) if not is_valid(n, k, buggy_off_by_one(n, k))]
print("số cặp (n, k) với n <= 8 bị sai:", len(bad), "| ví dụ:", bad[:5])
n, k = bad[0]
a = buggy_off_by_one(n, k)
print(f"(n={n}, k={k}):", a, "hiệu", diffs(a), "-> khác nhau:", len(set(diffs(a))), "(cần", k, ")")
assert bad

số cặp (n, k) với n <= 8 bị sai: 21 | ví dụ: [(3, 2), (4, 2), (4, 3), (5, 2), (5, 3)]
(n=3, k=2): [1, 2, 3] hiệu [1, 1] -> khác nhau: 1 (cần 2 )


### Lỗi 3 — Điền phần còn lại sai chiều (bỏ qua tính chẵn/lẻ của `k`)

Trong Cách 4, hướng điền phụ thuộc vào con trỏ nào vừa được dùng ở lượt zigzag cuối: `k` lẻ thì dùng `l`, `k` chẵn thì dùng `r`. Nếu luôn điền tăng từ `l`, thì khi `k` chẵn ta nhảy từ `r + 1` xuống `l` (cách xa), tạo thêm hiệu mới.

In [31]:
def buggy_always_increase(n, k):
    l, r = 1, n
    ans = []
    for i in range(k):
        if i % 2 == 0:
            ans.append(l); l += 1
        else:
            ans.append(r); r -= 1
    for i in range(k, n):               # BUG: luôn tăng từ l
        ans.append(l); l += 1
    return ans

bad = [(n, k) for n in range(2, 9) for k in range(1, n) if not is_valid(n, k, buggy_always_increase(n, k))]
print("các cặp sai:", bad[:6], "... (đều có k chẵn:", all(k % 2 == 0 for _, k in bad), ")")
n, k = 5, 2
a = buggy_always_increase(n, k)
print(f"(n={n}, k={k}):", a, "hiệu", diffs(a), "-> khác nhau:", len(set(diffs(a))), "(cần", k, ")")
assert bad

các cặp sai: [(4, 2), (5, 2), (6, 2), (6, 4), (7, 2), (7, 4)] ... (đều có k chẵn: True )
(n=5, k=2): [1, 5, 2, 3, 4] hiệu [4, 3, 1, 1] -> khác nhau: 3 (cần 2 )


### Lỗi 4 — Làm mất/trùng phần tử (không còn là hoán vị)

Nếu quên điền phần còn lại, hoặc để `l` và `r` cùng lấy một số khi gặp nhau, `answer` thiếu hay trùng số. Một hàm kiểm tra chỉ đếm hiệu (không kiểm tra hoán vị) sẽ **chấp nhận nhầm**. Luôn kiểm tra cả hai điều kiện.

In [32]:
def weak_check(n, k, ans):
    """Kiểm tra THIẾU: chỉ đếm hiệu, quên kiểm tra là hoán vị của 1..n."""
    return len(set(diffs(ans))) == k

def buggy_forget_fill(n, k):
    l, r, ans = 1, n, []
    for i in range(k):               # BUG: quên điền n - k phần tử còn lại
        if i % 2 == 0:
            ans.append(l); l += 1
        else:
            ans.append(r); r -= 1
    return ans

a = buggy_forget_fill(6, 4)
print(a, "| weak_check:", weak_check(6, 3, a), "| is_valid:", is_valid(6, 3, a), "(dài", len(a), "thay vì 6)")
print([1, 1, 1, 1], "| weak_check(4, 1):", weak_check(4, 1, [1, 1, 1, 1]), "| is_valid:", is_valid(4, 1, [1, 1, 1, 1]))
assert weak_check(4, 1, [1, 1, 1, 1]) and not is_valid(4, 1, [1, 1, 1, 1])

[1, 6, 2, 5] | weak_check: True | is_valid: False (dài 4 thay vì 6)
[1, 1, 1, 1] | weak_check(4, 1): True | is_valid: False


### Lỗi 5 — Quên rằng `k < n`: không thể có `k = n`

Chỉ có `n - 1` hiệu nên không bao giờ có `n` giá trị khác nhau. Một vét cạn "để chắc" cho `k = n` sẽ chạy hết `n!` hoán vị rồi không tìm thấy. Hãy kiểm chứng bằng oracle ở `n` nhỏ: mọi `k >= n` đều không có hoán vị hợp lệ.

In [33]:
for n in range(2, 8):
    for k in (n, n + 1):
        assert count_valid(n, k) == 0
print("với n <= 7, mọi hoán vị đều có tối đa n - 1 giá trị khác nhau; k = n hay n + 1 không bao giờ khả thi")
print("giá trị k lớn nhất thực sự đạt được ở n=6:", max(len(set(diffs(p))) for p in permutations(range(1, 7))))

với n <= 7, mọi hoán vị đều có tối đa n - 1 giá trị khác nhau; k = n hay n + 1 không bao giờ khả thi
giá trị k lớn nhất thực sự đạt được ở n=6: 5


## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** bài **dựng (construct)** một hoán vị với yêu cầu về hiệu/tổng liền kề, "return any" ⇒ nghĩ đến **mẫu zigzag** `lo, hi, lo+1, hi-1, ...`.
- Zigzag tạo ra các hiệu **phân biệt**, đơn điệu tạo ra hiệu **lặp** (`1`). Muốn đúng `k` giá trị: ghép **một đoạn zigzag + một đoạn đơn điệu**.
- Khi có nhiều đáp án: viết hàm kiểm tra `is_valid` và dùng nó trong mọi test; đừng so sánh cứng.
- Hãy quan sát bài nhỏ (`n <= 6`) bằng vét cạn để **đoán mẫu**, rồi chứng minh bằng đếm hiệu.
- Kiểm tra mọi `(n, k)` nhỏ bằng vòng lặp lồng; lỗi lệch 1 bộc lộ ngay ở các cặp nhỏ.
- Các bài cùng họ: **526. Beautiful Arrangement**, **932. Beautiful Array**, **1053. Previous Permutation With One Swap**, **484. Find Permutation**, **2191. Sort the Jumbled Numbers**.

## 15. Biến thể

### 15.1. Bắt đầu bằng `n` (đối xứng)

Nếu bắt buộc phần tử đầu là `n`, lật đối xứng toàn bộ: `y -> n + 1 - y`. Phép lật giữ nguyên mọi hiệu và vẫn là hoán vị.

In [34]:
def construct_start_with_n(n, k):
    return [n + 1 - y for y in Solution().constructArray(n, k)]

for n in range(2, 30):
    for k in range(1, n):
        a = construct_start_with_n(n, k)
        assert a[0] == n and is_valid(n, k, a)
print(construct_start_with_n(6, 3))

[6, 1, 5, 4, 3, 2]


### 15.2. Đảo thứ tự: đơn điệu trước, zigzag sau

Ta cũng có thể đặt đoạn đơn điệu **ở đầu** và zigzag trên `k + 1` số lớn nhất ở cuối: `1, 2, ..., n-k-1`, rồi zigzag trên `n-k..n`.

In [35]:
def construct_prefix_then_zigzag(n, k):
    ans = list(range(1, n - k))               # 1..n-k-1 tăng
    lo, hi = n - k, n
    for i in range(k + 1):                    # zigzag trên n-k..n (k+1 số)
        if i % 2 == 0:
            ans.append(lo); lo += 1
        else:
            ans.append(hi); hi -= 1
    return ans

for n in range(2, 60):
    for k in range(1, n):
        assert is_valid(n, k, construct_prefix_then_zigzag(n, k)), (n, k)
print(construct_prefix_then_zigzag(8, 3), "| hiệu", diffs(construct_prefix_then_zigzag(8, 3)))

[1, 2, 3, 4, 5, 8, 6, 7] | hiệu [1, 1, 1, 1, 3, 2, 1]


### 15.3. Đếm số hoán vị hợp lệ (bảng `n`, `k`) bằng vét cạn

Số đáp án tăng rất nhanh; đó là lý do đề cho "any" và ta cần dựng thay vì dò.

In [36]:
print("n \\ k | " + " ".join(f"{k:5d}" for k in range(1, 8)))
for n in range(2, 8):
    row = [count_valid(n, k) if k < n else 0 for k in range(1, 8)]
    print(f"{n:5d} | " + " ".join(f"{v:5d}" for v in row), "| tổng =", sum(row), "= n! =", __import__("math").factorial(n))
    assert sum(row) == __import__("math").factorial(n)

n \ k |     1     2     3     4     5     6     7
    2 |     2     0     0     0     0     0     0 | tổng = 2 = n! = 2
    3 |     2     4     0     0     0     0     0 | tổng = 6 = n! = 6
    4 |     2    18     4     0     0     0     0 | tổng = 24 = n! = 24
    5 |     2    46    64     8     0     0     0 | tổng = 120 = n! = 120
    6 |     2    98   400   196    24     0     0 | tổng = 720 = n! = 720
    7 |     2   178  1596  2456   776    32     0 | tổng = 5040 = n! = 5040


Mỗi hàng cộng lại đúng bằng `n!`, vì mỗi hoán vị có đúng một số hiệu khác nhau (từ `1` đến `n-1`). Hàng `n = 7` cho thấy với `k` nhỏ hoặc lớn số đáp án còn ít, ở giữa thì rất nhiều.

### 15.4. Dựng đáp án với **tập hiệu** cho trước là `{1, ..., k}` (hơn yêu cầu)

Cách 3 thực chất cho tập hiệu đúng bằng `{1, 2, ..., k}`; Cách 4 cho `{1} ∪ {n-k+1, ..., n-1}`. Hàm sau kiểm tra điều đó.

In [37]:
for n in range(2, 40):
    for k in range(1, n):
        s3 = set(diffs(construct_small_zigzag(n, k)))
        s4 = set(diffs(construct_two_pointers(n, k)))
        assert s3 == set(range(1, k + 1))
        assert s4 == {1} | set(range(n - k + 1, n))
print("Cách 3 cho tập hiệu {1..k}; Cách 4 cho {1} ∪ {n-k+1..n-1}")
print(sorted(set(diffs(construct_small_zigzag(9, 4)))), sorted(set(diffs(construct_two_pointers(9, 4)))))

Cách 3 cho tập hiệu {1..k}; Cách 4 cho {1} ∪ {n-k+1..n-1}
[1, 2, 3, 4] [1, 6, 7, 8]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi hoán vị, đếm hiệu | `O(n! · n)` | `O(n)` | Chỉ làm oracle với `n <= 8` |
| 2. Quay lui | Dựng dần, cắt nhánh khi quá `k` hiệu | Hàm mũ | `O(n)` | Học cắt nhánh; `n` nhỏ |
| 3. Zigzag nhỏ + nối tăng | Zigzag `1..k+1` rồi `k+2..n` | `O(n)` | `O(n)` | Dễ chứng minh, tập hiệu đẹp `{1..k}` |
| 4. Hai con trỏ (nộp) | Zigzag `k` số đầu trên `1..n`, điền đơn điệu | `O(n)` | `O(n)` | Mặc định: ngắn, chắc tay |
| 5. Đệ quy quy nạp | `(n,k)` → `(n-1,k-1)` bằng phép lật | `O(n · k)` | `O(n + k)` | Để chứng minh bằng quy nạp |

Cách 3 và 4 đều tuyến tính; cách 4 ngắn gọn hơn nhưng cần để ý tính chẵn lẻ của `k`.

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(diffs([1, 4, 2, 3]))
print(len(set(diffs([1, 4, 2, 3]))))
print(is_valid(4, 2, [2, 1, 4, 3]))
```

In [38]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Dùng thuật toán hai con trỏ (Cách 4), hãy tính tay `constructArray(7, 3)`. Ghi ra dãy đáp án, danh sách hiệu, và số giá trị khác nhau của hiệu.

In [39]:
# Bài 2: điền kết quả tính tay
hand_answer = None        # list đáp án
hand_diffs = None         # list các hiệu liền kề
hand_distinct = None      # số giá trị khác nhau của hiệu

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 3 (zigzag trong `1..k+1` rồi nối tăng).

In [40]:
def exercise_3_small_zigzag(n: int, k: int) -> List[int]:
    ans = []
    l, r = 1, k + 1
    # TODO 1: lặp k + 1 lần; lượt chẵn lấy l (rồi l += 1), lượt lẻ lấy r (rồi r -= 1)
    # TODO 2: nối các số k+2 .. n theo thứ tự tăng
    return None


# for n, k in [(3, 1), (3, 2), (7, 3), (10, 9)]:
#     assert is_valid(n, k, exercise_3_small_zigzag(n, k))

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_two_pointers(n, k)`: zigzag `k` phần tử đầu rồi điền `n - k` phần tử còn lại đơn điệu **đúng chiều theo tính chẵn lẻ của `k`**.

In [41]:
def exercise_4_two_pointers(n: int, k: int) -> List[int]:
    # TODO
    pass


# for n, k in [(3, 1), (3, 2), (7, 3), (7, 4), (10, 9)]:
#     assert is_valid(n, k, exercise_4_two_pointers(n, k))

### Bài 5 — Mini challenge: phần tử đầu phải là `n`, và dãy hiệu phải **không tăng** sau đoạn zigzag (khó)

Viết `exercise_5_start_with_n(n, k)` trả về một đáp án hợp lệ của `(n, k)` mà **phần tử đầu tiên bằng `n`**. Sau đó viết thêm `exercise_5_count_distinct_max(n)` trả về số giá trị khác nhau **lớn nhất** có thể đạt được của dãy hiệu qua mọi hoán vị của `1..n` (với `n >= 2`).

```python
exercise_5_start_with_n(5, 2)[0] -> 5
exercise_5_count_distinct_max(2) -> 1
exercise_5_count_distinct_max(6) -> 5
```

In [42]:
def exercise_5_start_with_n(n: int, k: int) -> List[int]:
    # TODO
    pass


def exercise_5_count_distinct_max(n: int) -> int:
    # TODO
    pass


# for n in range(2, 12):
#     for k in range(1, n):
#         a = exercise_5_start_with_n(n, k)
#         assert a[0] == n and is_valid(n, k, a)
# assert exercise_5_count_distinct_max(2) == 1
# assert exercise_5_count_distinct_max(6) == 5

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `diffs` lấy hiệu tuyệt đối của từng cặp liền kề.
- Mức 2: các cặp của `[1,4,2,3]` là `(1,4), (4,2), (2,3)`.
- Mức 3: dòng 1 là `[3, 2, 1]`; dòng 2 là `3`; dòng 3: `[2,1,4,3]` có hiệu `[1,3,1]` → 2 giá trị `{1, 3}` và là hoán vị của `1..4` nên `True`.

**Bài 2**
- Mức 1: `n = 7`, `k = 3`; zigzag lấy 3 phần tử, `k` lẻ nên điền tăng từ `l`.
- Mức 2: zigzag `[1, 7, 2]`; sau đó `l = 3`, điền `3, 4, 5, 6`.
- Mức 3: đáp án `[1,7,2,3,4,5,6]`; hiệu `[6,5,1,1,1,1]`; `3` giá trị khác nhau `{6, 5, 1}`.

**Bài 3**
- Mức 1: giống mục 7.2.
- Mức 2: `for i in range(k + 1): if i % 2 == 0: ans.append(l); l += 1 else: ans.append(r); r -= 1`.
- Mức 3: kết thúc `ans.extend(range(k + 2, n + 1))` rồi `return ans`.

**Bài 4**
- Mức 1: dùng `l = 1`, `r = n` và hai vòng `for`.
- Mức 2: vòng 2 chạy `n - k` lần; nếu `k % 2 == 0` lấy `r` (giảm), ngược lại lấy `l` (tăng).
- Mức 3: kiểm tra `n = 5, k = 2`: zigzag `[1, 5]` rồi điền giảm `4, 3, 2`.

**Bài 5**
- Mức 1: lật đối xứng `y -> n + 1 - y` giữ nguyên hiệu.
- Mức 2: đáp án nộp bắt đầu bằng `1` nên sau khi lật bắt đầu bằng `n`.
- Mức 3: `count_distinct_max(n) = n - 1` (có `n - 1` hiệu và zigzag toàn bộ làm chúng khác nhau).

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [43]:
ans1 = [
    diffs([1, 4, 2, 3]),
    len(set(diffs([1, 4, 2, 3]))),
    is_valid(4, 2, [2, 1, 4, 3]),
]
print(ans1)
assert ans1 == [[3, 2, 1], 3, True]

[[3, 2, 1], 3, True]


### Đáp án Bài 2

In [44]:
ans2_answer = [1, 7, 2, 3, 4, 5, 6]
ans2_diffs = [6, 5, 1, 1, 1, 1]
ans2_distinct = 3
assert Solution().constructArray(7, 3) == ans2_answer
assert diffs(ans2_answer) == ans2_diffs
assert len(set(ans2_diffs)) == ans2_distinct
assert is_valid(7, 3, ans2_answer)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [45]:
def ans3_small_zigzag(n: int, k: int) -> List[int]:
    ans = []
    l, r = 1, k + 1
    for i in range(k + 1):
        if i % 2 == 0:
            ans.append(l)
            l += 1
        else:
            ans.append(r)
            r -= 1
    ans.extend(range(k + 2, n + 1))
    return ans

for n, k in [(3, 1), (3, 2), (7, 3), (10, 9)]:
    assert is_valid(n, k, ans3_small_zigzag(n, k))
for n in range(2, 40):
    for k in range(1, n):
        assert is_valid(n, k, ans3_small_zigzag(n, k))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [46]:
def ans4_two_pointers(n: int, k: int) -> List[int]:
    l, r = 1, n
    ans = []
    for i in range(k):
        if i % 2 == 0:
            ans.append(l)
            l += 1
        else:
            ans.append(r)
            r -= 1
    for i in range(k, n):
        if k % 2 == 0:
            ans.append(r)
            r -= 1
        else:
            ans.append(l)
            l += 1
    return ans

for n in range(2, 40):
    for k in range(1, n):
        assert is_valid(n, k, ans4_two_pointers(n, k))
assert ans4_two_pointers(5, 2) == [1, 5, 4, 3, 2]
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [47]:
def ans5_start_with_n(n: int, k: int) -> List[int]:
    return [n + 1 - y for y in ans4_two_pointers(n, k)]

def ans5_count_distinct_max(n: int) -> int:
    return n - 1

for n in range(2, 12):
    for k in range(1, n):
        a = ans5_start_with_n(n, k)
        assert a[0] == n and is_valid(n, k, a)
assert ans5_start_with_n(5, 2)[0] == 5
assert ans5_count_distinct_max(2) == 1
assert ans5_count_distinct_max(6) == 5
for n in range(2, 8):
    assert max(len(set(diffs(p))) for p in permutations(range(1, n + 1))) == ans5_count_distinct_max(n)
print("Bài 5 OK (đối chiếu vét cạn n <= 7)")

Bài 5 OK (đối chiếu vét cạn n <= 7)


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def constructArray(self, n: int, k: int) -> List[int]:
        l, r = 1, n
        ans = []
        for i in range(k):                 # zigzag k phần tử đầu: 1, n, 2, n-1, ...
            if i % 2 == 0:
                ans.append(l); l += 1
            else:
                ans.append(r); r -= 1
        for i in range(k, n):              # điền đơn điệu theo chiều con trỏ vừa dừng
            if k % 2 == 0:
                ans.append(r); r -= 1      # k chẵn: vừa lấy r -> tiếp tục giảm
            else:
                ans.append(l); l += 1      # k lẻ: vừa lấy l -> tiếp tục tăng
        return ans
```

Nhớ ba ý: **zigzag tạo hiệu mới (mỗi phần tử một hiệu), đơn điệu chỉ tạo hiệu 1**, **zigzag `k` phần tử + điền đơn điệu = đúng `k` hiệu khác nhau**, **bài nhiều đáp án thì test bằng `is_valid`**.

### Checklist trước khi submit

- [ ] Kết quả là hoán vị của `1..n` (đủ `n` phần tử, không trùng).
- [ ] Đã kiểm tra tính chất bằng `is_valid`, không so cứng với đáp án mẫu.
- [ ] Số phần tử zigzag đúng công thức (`k` trên toàn dải, hoặc `k + 1` trên đoạn `1..k+1`).
- [ ] Phần điền đơn điệu đi **đúng chiều** theo tính chẵn lẻ của `k`.
- [ ] Đã thử `k = 1`, `k = n - 1`, `n = 2`, `k` chẵn và lẻ.
- [ ] Time `O(n)`, Space `O(n)` cho kết quả.